In [20]:
import mediapipe as mp
import cv2
mp_pose = mp.solutions.pose
attention_dot = [n for n in range(start_dot, 29)]

In [21]:
# 라인 그리기
if start_dot == 11:
    """몸 부분만"""
    draw_line = [[11, 13], [13, 15], [15, 21], [15, 19], [15, 17], [17, 19], \
                [12, 14], [14, 16], [16, 22], [16, 20], [16, 18], [18, 20], \
                [23, 25], [25, 27], [24, 26], [26, 28], [11, 12], [11, 23], \
                [23, 24], [12, 24]]
    print('Pose : Only Body')

else:
    """얼굴 포함"""
    draw_line = [[11, 13], [13, 15], [15, 21], [15, 19], [15, 17], [17, 19], \
                [12, 14], [14, 16], [16, 22], [16, 20], [16, 18], [18, 20], \
                [23, 25], [25, 27], [24, 26], [26, 28], [11, 12], [11, 23], \
                [23, 24], [12, 24], [9, 10], [0, 5], [0, 2], [5, 8], [2, 7]]
    print('Pose : Face + Body')

Pose : Only Body


In [22]:
# 설정값
frame_length = 30  # LSTM에 필요한 프레임 수
n_CONFIDENCE = 0.3  # mediapipe 신뢰도
y_CONFIDENCE = 0.3  # yolo 신뢰도
start_dot = 11      # 11이면 어깨~발, 0이면 얼굴 포함

# 모델 불러오기
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo_model = YOLO('yolov8s.pt')
yolo_model.to(device)

# MediaPipe 설정
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1,
                    enable_segmentation=False, min_detection_confidence=n_CONFIDENCE)


# 주요 포인트 (몸통 기준)
attention_dot = [i for i in range(start_dot, 29)]


def get_skeleton(video_path, attention_dot, draw_line):
    xy_list_list, xy_list_list_flip = [], []

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ 영상 열기 실패:", video_path)
        return False, False

    while True:
        ret, img = cap.read()
        if not ret:
            break

        # YOLO 입력 크기
        img = cv2.resize(img, (640, 640))
        results = yolo_model.predict(img, classes=[0], verbose=False)  # class 0: person

        boxes = results[0].boxes.cpu().numpy()
        if boxes.shape[0] == 0:
            continue

        for box in boxes:
            x1, y1, x2, y2, conf = int(box.xyxy[0][0])-10, int(box.xyxy[0][1]), int(box.xyxy[0][2])+10, int(box.xyxy[0][3]), box.conf[0]
            x1 = max(0, x1)
            x2 = min(639, x2)
            y1 = max(0, y1)
            y2 = min(639, y2)

            if conf < y_CONFIDENCE:
                continue

            cropped = img[y1:y2, x1:x2]
            pose_result = pose.process(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
            if not pose_result.pose_landmarks:
                continue

            xy_list, xy_list_flip = [], []
            draw_line_dic = {}

            for idx, landmark in enumerate(pose_result.pose_landmarks.landmark):
                if idx in attention_dot:
                    x = landmark.x
                    y = landmark.y
                    xy_list += [x, y]
                    xy_list_flip += [1 - x, y]
                    draw_line_dic[idx] = (int(x * (x2 - x1)), int(y * (y2 - y1)))

            if len(xy_list) != len(attention_dot) * 2:
                print('⚠️ landmark 수 불일치 발생')
                continue

            xy_list_list.append(xy_list)
            xy_list_list_flip.append(xy_list_flip)

            # 시각화용 draw_line 연결 (주석 해제 시 확인 가능)
            # for line in draw_line:
            #     pt1 = draw_line_dic.get(line[0])
            #     pt2 = draw_line_dic.get(line[1])
            #     if pt1 and pt2:
            #         cropped = cv2.line(cropped, pt1, pt2, (0, 255, 0), 2)
            # cv2.imshow("Pose", cropped)
            # cv2.waitKey(1)

    cap.release()
    cv2.destroyAllWindows()

    # 프레임 부족 시 보완
    if len(xy_list_list_flip) < 15:
        return False, False
    elif len(xy_list_list_flip) < frame_length:
        last = xy_list_list[-1]
        last_flip = xy_list_list_flip[-1]
        for _ in range(frame_length - len(xy_list_list_flip)):
            xy_list_list.append(last)
            xy_list_list_flip.append(last_flip)

    return xy_list_list, xy_list_list_flip

I0000 00:00:1749103914.372934  601137 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1749103914.376384  728028 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


In [23]:
import os
import numpy as np
from tqdm import tqdm

def build_dataset_from_folder(folder_path, label, attention_dot, draw_line):
    X, y = [], []
    for fname in tqdm(os.listdir(folder_path)):
        if fname.endswith(".mp4"):
            video_path = os.path.join(folder_path, fname)
            xy_list, xy_list_flip = get_skeleton(video_path, attention_dot, draw_line)
            if xy_list and len(xy_list) == 30:  # 30프레임 확인
                X.append(xy_list)
                y.append(label)
            if xy_list_flip and len(xy_list_flip) == 30:  # flip도 확인
                X.append(xy_list_flip)
                y.append(label)
    return X, y


# 정상 / 이상 각각 처리
normal_path = "./result/normal"
abnormal_path = "./result/abnormal"

X_normal, y_normal = build_dataset_from_folder(normal_path, 0, attention_dot, draw_line)
X_abnormal, y_abnormal = build_dataset_from_folder(abnormal_path, 1, attention_dot, draw_line)

# 결합
X_total = np.array(X_normal + X_abnormal)
y_total = np.array(y_normal + y_abnormal)

print("✅ 데이터 shape:", X_total.shape, y_total.shape)


100%|██████████| 105/105 [05:34<00:00,  3.18s/it]

✅ 데이터 shape: (276, 30, 36) (276,)


In [25]:
import collections
print("클래스 비율:", collections.Counter(y_total))


클래스 비율: Counter({np.int64(0): 138, np.int64(1): 138})


In [24]:
np.save("X_pose.npy", X_total)
np.save("y_pose.npy", y_total)
print("💾 저장 완료: X_pose.npy, y_pose.npy")


💾 저장 완료: X_pose.npy, y_pose.npy
